# Detection of Suspicious Transactions via Anomaly Detection (AML Project)

**Course:** Data Science  
**Author:** Slavena Georgieva
**Date:** April 2026

---

## 1. Introduction

Money laundering is the process of concealing the origins of illegally obtained money.
According to the United Nations, between **800 billion and 2 trillion** are laundered
globally each year — representing 2–5% of global GDP.

Traditional rule-based Anti-Money Laundering (AML) systems flag transactions based on
fixed thresholds (e.g. transactions above $10,000). While simple, these systems produce
high false-positive rates and are easily circumvented by criminals who deliberately
structure transactions to stay below detection thresholds.

This project takes a **statistical and machine learning approach** — using Anomaly Detection
to identify suspicious transactions based on their deviation from normal behaviour,
rather than fixed rules.

### Objectives
- Perform Exploratory Data Analysis (EDA) on two independent AML datasets
- Test a statistical hypothesis about transaction amounts
- Apply and compare three anomaly detection methods: Z-score, IQR, and Isolation Forest

## 2. Research Hypothesis

**Null Hypothesis (H₀):**  
There is no statistically significant difference between the transaction amounts
of normal and suspicious transactions.

**Alternative Hypothesis (H₁):**  
Suspicious transactions have significantly higher amounts compared to normal transactions.

$$H_0: \mu_{suspicious} = \mu_{normal}$$

$$H_1: \mu_{suspicious} > \mu_{normal}$$

This is a **one-tailed test** (right-tailed), since H₁ is directional — we expect
suspicious transactions to be *higher*, not just different.

We use a significance level of **α = 0.05**.

## 3. Data Sources & Description

### Sampling Strategy

Due to file size constraints (SAML-D: ~200MB, IBM HI-Large: ~15GB),
this project works with stratified samples of 10,000 rows from each dataset.

**Stratified sampling** ensures that the proportion of suspicious vs. normal
transactions is preserved from the original dataset — avoiding a sample
with zero suspicious transactions.

The samples are saved as separate CSV files and committed to the repository.
Full datasets are available on Kaggle (see README).

In [5]:
import pandas as pd
import numpy as np

# ── Load full SAML-D ───────────────────────────────────
saml_full = pd.read_csv('data/SAML-D.csv')

print(f"Full SAML-D shape: {saml_full.shape}")
print(f"Columns: {saml_full.columns.tolist()}")
print("\nLabel distribution:")
print(saml_full.iloc[:, -1].value_counts())


Full SAML-D shape: (7014020, 12)
Columns: ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']

Label distribution:
Normal_Small_Fan_Out      2566088
Normal_Fan_Out            1707805
Normal_Fan_In             1562331
Normal_Group               389802
Normal_Cash_Withdrawal     225955
Normal_Cash_Deposits       164683
Normal_Periodical          155796
Normal_Plus_Mutual          95962
Normal_Mutual               92400
Normal_Foward               30941
Normal_single_large         15254
Structuring                  1316
Cash_Withdrawal               934
Deposit-Send                  686
Smurfing                      669
Layered_Fan_In                484
Layered_Fan_Out               396
Bipartite                     328
Stacked Bipartite             310
Cycle                         299
Fan_In                        275
Behavioural_Cha

In [6]:
# ── Identify label column ──────────────────────────────
label_col = saml_full.columns[-1]  # usually last column

# ── Stratified sample (preserves suspicious ratio) ─────
saml_sample = saml_full.groupby(label_col, group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(10_000 * len(x) / len(saml_full))),
        random_state=42
    )
).reset_index(drop=True)

print(f"Sample shape: {saml_sample.shape}")
print("\nLabel distribution in sample:")
print(saml_sample[label_col].value_counts())

# ── Save sample ─────────────────────────────────────────
saml_sample.to_csv('data/saml_sample.csv', index=False)
print("\n✓ Saved: data/saml_sample.csv")


Sample shape: (9986, 12)

Label distribution in sample:
Normal_Small_Fan_Out      3658
Normal_Fan_Out            2434
Normal_Fan_In             2227
Normal_Group               555
Normal_Cash_Withdrawal     322
Normal_Cash_Deposits       234
Normal_Periodical          222
Normal_Plus_Mutual         136
Normal_Mutual              131
Normal_Foward               44
Normal_single_large         21
Cash_Withdrawal              1
Structuring                  1
Name: Laundering_type, dtype: int64

✓ Saved: data/saml_sample.csv


In [7]:
# ── Read IBM dataset in chunks (file is huge) ───────────
print("Reading IBM file in chunks... (this may take 1–2 minutes)")

chunks = []
for chunk in pd.read_csv(
    'data/HI-Large_Trans.csv',
    chunksize=100_000,
    dtype={'Account': str, 'Account.1': str},
    low_memory=False
):
    chunks.append(chunk)
    if len(chunks) == 5:  # read first ~500k rows
        break

ibm_full = pd.concat(chunks).reset_index(drop=True)

print(f"Loaded shape: {ibm_full.shape}")
print(f"Columns: {ibm_full.columns.tolist()}")


Reading IBM file in chunks... (this may take 1–2 minutes)
Loaded shape: (500000, 11)
Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


In [8]:
# ── Identify label column ───────────────────────────────
label_col_ibm = 'Is Laundering'  # change if different

print("Label distribution:")
print(ibm_full[label_col_ibm].value_counts())

# ── Stratified sample ───────────────────────────────────
ibm_sample = ibm_full.groupby(label_col_ibm, group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(10_000 * len(x) / len(ibm_full))),
        random_state=42
    )
).reset_index(drop=True)

print(f"\nSample shape: {ibm_sample.shape}")
print("\nLabel distribution in sample:")
print(ibm_sample[label_col_ibm].value_counts())

# ── Save sample ─────────────────────────────────────────
ibm_sample.to_csv('data/ibm_sample.csv', index=False)
print("\n✓ Saved: data/ibm_sample.csv")


Label distribution:
0    499965
1        35
Name: Is Laundering, dtype: int64

Sample shape: (9999, 11)

Label distribution in sample:
0    9999
Name: Is Laundering, dtype: int64

✓ Saved: data/ibm_sample.csv
